In [ ]:
import os
lat = 34.499984
lon = -4.708586
site_name = 'test_site_name'
client = '34,499984_-4,708586'
surface_tilt = 30
startDate = "2005-01-01"
endDate   = "2025-12-31"
job_id = None  
selected_files = []


In [ ]:
# Status updater — uses HTTP so no Django setup needed
import requests

def update_job(status, message=''):
    if job_id:
        try:
            requests.post(
                #  CORRECT - Docker service name
                # f'http://web:8000/api/tmy/update/{job_id}/',
                f'http://web:8000/api/tmy/update-internal/{job_id}/',
                json={'status': status, 'message': message},
                timeout=5
            )
            print(f"[Job #{job_id}] {status}: {message}")
        except Exception as e:
            print(f"[update_job failed] {e}")
    else:
        print(f"[No job_id] {status}: {message}")

In [106]:
import pandas as pd
import numpy as np

In [107]:
TMYs_folder = 'TMYs'
TMYs_path = os.path.join(TMYs_folder,site_name)
if not os.path.exists(TMYs_path):
    os.makedirs(TMYs_path)

In [ ]:
import os

downloads_dir = os.path.join(TMYs_folder, site_name, "downloads")
os.makedirs(downloads_dir, exist_ok=True)

print("Selected files:", selected_files)

### Configuring cdsapi

In [ ]:
api_token = {
    'era5': {
        'url': 'https://cds.climate.copernicus.eu/api'
    },
    'cams': {
        'url': 'https://ads.atmosphere.copernicus.eu/api'
    }
}
# def update_cdsapirc(api_type, file_path=r"C:\Users\DELL\.cdsapirc"):

# def update_cdsapirc(api_type, file_path=None):
#     if file_path is None:
#         file_path = os.path.expanduser("~/.cdsapirc")
#     """
#     Updates the .cdsapirc file with the specified API URL.

#     Parameters:
#         api_type (str): The API type to use ('era5' or 'cams').
#         file_path (str): The path to the .cdsapirc file.
#     """
#     # Define the API URLs
#     api_url = {
#         'era5': {
#             'url': 'https://cds.climate.copernicus.eu/api'
#         },
#         'cams': {
#             'url': 'https://ads.atmosphere.copernicus.eu/api'
#         }
#     }

#     # Validate the input
#     if api_type not in api_url:
#         raise ValueError("Invalid API type. Please choose 'era5' or 'cams'.")

#     try:
#         # Read the file content
#         with open(file_path, 'r') as file:
#             lines = file.readlines()

#         # Update the lines based on the selected API type
#         updated_lines = [] 
#         for line in lines:
#             if "url:" in line:
#                 updated_lines.append(f"url: {api_url[api_type]['url']}\n")
#             else:
#                 updated_lines.append(line)

#         # Write the updated content back to the file
#         with open(file_path, 'w') as file:
#             file.writelines(updated_lines)

#         print(f"The .cdsapirc file has been updated to use '{api_type}' API.")
#     except FileNotFoundError:
#         print(f"The file '{file_path}' does not exist.")
#     except Exception as e:
#         print(f"An error occurred: {e}")


def update_cdsapirc(api_type, file_path=None):
    if file_path is None:
        file_path = os.path.expanduser("~/.cdsapirc")
    
    api_url = {
        'era5': {'url': 'https://cds.climate.copernicus.eu/api'},
        'cams': {'url': 'https://ads.atmosphere.copernicus.eu/api'}
    }

    if api_type not in api_url:
        raise ValueError("Invalid API type. Choose 'era5' or 'cams'.")

    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()

        updated_lines = []
        for line in lines:
            if "url:" in line:
                updated_lines.append(f"url: {api_url[api_type]['url']}\n")
            else:
                updated_lines.append(line)

        with open(file_path, 'w') as file:
            file.writelines(updated_lines)

        print(f"Updated .cdsapirc to use '{api_type}' API.")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error: {e}")

In [109]:
update_cdsapirc('cams')

The .cdsapirc file has been updated to use 'cams' API.


### Downloading cams data

In [ ]:
import cdsapi

update_job('downloading_cams', 'Downloading CAMS solar radiation data from Copernicus ADS...')
c = cdsapi.Client()

c.retrieve(
    "cams-solar-radiation-timeseries",
    {
    "sky_type": "observed_cloud",
    "location": {"longitude": lon, "latitude": lat},
    "altitude": ["-999."],
    "date": [f'{startDate}/{endDate}'],
    "time_step": "1hour",
    "time_reference": "universal_time",
    "format": "csv"
    },
    f'{TMYs_folder}/{site_name}/{site_name}_cams.csv')
# # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'processing_data'
#     j.status_message = 'All data downloaded. Processing and merging ERA5 + CAMS datasets...'
#     j.save()


2026-05-18 11:08:16,508 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
2026-05-18 11:08:16,809 INFO Request ID is 2bbd0b16-b1d2-4b45-b029-9b3ef694d426
2026-05-18 11:08:16,931 INFO status has been updated to accepted
2026-05-18 11:08:50,652 INFO status has been updated to running
2026-05-18 11:10:14,722 INFO status has been updated to successful


2c622b1c386d7583f087ef95c9e2e003.csv:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

'TMYs/test_site_name/test_site_name_cams.csv'

In [ ]:
update_job('downloading_era5', f'Downloading ERA5 climate data {startDate[:4]} to {endDate[:4]}...')
update_cdsapirc('era5')
# # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'downloading_era5'
#     j.status_message = 'Connecting to Copernicus CDS and downloading ERA5 data...'
#     j.save()

The .cdsapirc file has been updated to use 'era5' API.


### Extracting ERA5 Time series

In [ ]:
import xarray as xr
import glob
from tqdm import tqdm
from metpy.calc import wind_speed, wind_direction, relative_humidity_from_dewpoint
from metpy.units import units

def round_to_quarter(value):
    return round(value / 0.25) * 0.25

lat_rounded = round_to_quarter(lat)
lon_rounded = round_to_quarter(lon)

c = cdsapi.Client()

c.retrieve(
    "reanalysis-era5-single-levels-timeseries",
    {
    "location": {"longitude": lon_rounded, "latitude": lat_rounded},
    "date": [f'{startDate}/{endDate}'],
    "data_format": "csv",
    "variable": [
        "2m_dewpoint_temperature",
        "mean_sea_level_pressure",
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "100m_u_component_of_wind",
        "100m_v_component_of_wind",
        "surface_solar_radiation_downwards"
    ],
    },
    f'{TMYs_folder}/{site_name}/{site_name}_era5.csv')
#     # STATUS UPDATE
# if 'job_id' in vars() and job_id:
#     from tmy_app.models import TMYJob
#     j = TMYJob.objects.get(id=job_id)
#     j.status = 'downloading_cams'
#     j.status_message = 'ERA5 download complete. Now downloading CAMS solar data...'
#     j.save()

2026-05-18 11:10:18,639 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
2026-05-18 11:10:18,965 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernic

979328900f39a271ec6807b8eb60e638.zip:   0%|          | 0.00/7.04M [00:00<?, ?B/s]

'TMYs/test_site_name/test_site_name_era5.csv'

In [113]:
import zipfile
output_path = f"{TMYs_folder}/{site_name}/{site_name}_era5.csv"
import os

desired_name = f"{site_name}_era5.csv"
folder_path = os.path.dirname(output_path)

if zipfile.is_zipfile(output_path):
    print("ZIP detected. Extracting...")

    with zipfile.ZipFile(output_path, 'r') as z:
        extracted_files = z.namelist()
        z.extractall(folder_path)

    os.remove(output_path)

    # Rename extracted file to your desired name
    extracted_file_path = os.path.join(folder_path, extracted_files[0])
    final_path = os.path.join(folder_path, desired_name)

    os.rename(extracted_file_path, final_path)

    print(f"File renamed to: {desired_name}")

else:
    print("Normal CSV file.")


ZIP detected. Extracting...
File renamed to: test_site_name_era5.csv


In [114]:
era5_point = pd.read_csv(rf"{TMYs_folder}/{site_name}/{site_name}_era5.csv")

In [115]:
import glob
from tqdm import tqdm
from metpy.calc import wind_speed, wind_direction, relative_humidity_from_dewpoint
from metpy.units import units

era5_point['t2m_C'] = era5_point['t2m'] - 273.15
era5_point['d2m_C'] = era5_point['d2m'] - 273.15
era5_point['msl_hpa'] = era5_point['msl'] / 100

era5_point['wind_speed'] = wind_speed(era5_point['u10'].values * units.meter / units.second, 
                                    era5_point['v10'].values * units.meter / units.second).magnitude

era5_point['wind_direction'] = wind_direction(era5_point['u10'].values * units.meter / units.second, 
                                            era5_point['v10'].values * units.meter / units.second).magnitude

era5_point['relative_humidity'] = relative_humidity_from_dewpoint(
        era5_point['t2m_C'].values * units.degC,
        era5_point['d2m_C'].values * units.degC
    ).magnitude * 100  # Convert to percentage

filtred_era5_point = era5_point[['valid_time','t2m_C', 'd2m_C', 'wind_speed', 'wind_direction', 'relative_humidity', 'msl_hpa', 'ssrd']]

ERA5 = filtred_era5_point.rename(columns={
        'valid_time': 'time',
        't2m_C': 'Temperature',
        'd2m_C': 'Dew Point',
        'msl_hpa': 'Pressure',
        'relative_humidity': 'Relative Humidity',
        'wind_speed': 'Wind Speed',
        'wind_direction': 'Wind Direction',
        'ssrd': 'Surface solar radiation downwards'
    })

    # Save to CSV
ERA5.to_csv(f"{TMYs_folder}/{site_name}/era5_{site_name}.csv", index=False)
print("\nTime series extraction complete. Data saved.")


Time series extraction complete. Data saved.


In [116]:
era5 = pd.read_csv(rf"{TMYs_folder}/{site_name}/era5_{site_name}.csv")

In [ ]:
if "ERA5" in selected_files:
    era5_point.to_csv(
        f"{downloads_dir}/ERA5.csv",
        index=False
    )

In [117]:
# Assuming your DataFrame is named df
era5['datetime'] = pd.to_datetime(era5['time'])

# Drop the original columns if you no longer need them
era5 = era5.drop(columns=['time'])

era5['datetime'] = pd.to_datetime(era5['datetime'])
era5.set_index('datetime', inplace = True)

In [118]:
era5

,Temperature,Dew Point,Wind Speed,Wind Direction,Relative Humidity,Pressure,Surface solar radiation downwards
datetime,,,,,,,
2005-01-01 00:00:00,5.05932,0.73614,2.298872,56.981728,73.594401,1030.17190,0.0
2005-01-01 01:00:00,4.81140,0.54116,2.244497,57.205445,73.830649,1030.20750,0.0
2005-01-01 02:00:00,4.25400,0.15070,2.140901,58.476877,74.626884,1030.02500,0.0
2005-01-01 03:00:00,4.41525,0.05844,2.057307,60.135257,73.295890,1030.00940,0.0
2005-01-01 04:00:00,4.68185,0.07840,1.962189,58.084258,72.045699,1029.86560,0.0
...,...,...,...,...,...,...,...
2025-12-31 19:00:00,11.67855,10.44012,1.166920,87.080577,92.107605,1018.84250,0.0
2025-12-31 20:00:00,11.91360,11.04520,1.159807,85.053442,94.416422,1018.95190,0.0
2025-12-31 21:00:00,11.59240,10.93966,0.991091,78.740584,95.766215,1019.39125,0.0


In [119]:
cams = pd.read_csv(rf"{TMYs_folder}/{site_name}/{site_name}_cams.csv", skiprows=42, sep=';')
cams['datetime'] = pd.to_datetime(cams['# Observation period'].str.split('/').str[0])
cams.set_index('datetime', inplace=True)
cams = cams[['TOA','Clear sky GHI','Clear sky BHI','Clear sky DHI', 'Clear sky BNI', 'GHI','DHI','BNI']]
cams.rename(columns={'BNI': 'DNI'}, inplace=True)

In [ ]:
if "CAMS" in selected_files:
    cams.to_csv(
        f"{downloads_dir}/CAMS.csv",
        index=False
    )

In [ ]:
update_job('processing_data', 'Processing and merging ERA5 + CAMS datasets...')

In [120]:
dataset = pd.concat([cams, era5], axis=1)

In [121]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 184080 entries, 2005-01-01 00:00:00 to 2025-12-31 23:00:00
Data columns (total 15 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   TOA                                184080 non-null  float64
 1   Clear sky GHI                      184080 non-null  float64
 2   Clear sky BHI                      184080 non-null  float64
 3   Clear sky DHI                      184080 non-null  float64
 4   Clear sky BNI                      184080 non-null  float64
 5   GHI                                184023 non-null  float64
 6   DHI                                184023 non-null  float64
 7   DNI                                184023 non-null  float64
 8   Temperature                        184080 non-null  float64
 9   Dew Point                          184080 non-null  float64
 10  Wind Speed                         184080 non-null  float64
 11  Wind 

In [122]:
dataset.to_csv(f'{TMYs_folder}/{site_name}/dataset_{site_name}.csv', index=True)

In [ ]:
if "Temperature" in selected_files:
    dataset[['Temperature']].to_csv(
        f"{downloads_dir}/temperature.csv"
    )

if "Wind Speed" in selected_files:
    dataset[['Wind Speed']].to_csv(
        f"{downloads_dir}/wind_speed.csv"
    )

if "Relative Humidity" in selected_files:
    dataset[['Relative Humidity']].to_csv(
        f"{downloads_dir}/humidity.csv"
    )

if "GHI" in selected_files:
    dataset[['GHI']].to_csv(
        f"{downloads_dir}/ghi.csv"
    )

if "DNI" in selected_files:
    dataset[['DNI']].to_csv(
        f"{downloads_dir}/dni.csv"
    )

In [123]:
daily_stats = dataset.resample('D').agg({
    'Temperature': ['min', 'max'],
    'Dew Point': ['min', 'max'],
    'Wind Speed': 'max'
})
daily_stats.columns = ['min_temp', 'max_temp', 'min_dew', 'max_dew', 'max_wind_speed']
daily_stats.reset_index(inplace=True)
dataset = dataset.reset_index().merge(daily_stats, left_on=dataset.index.date, right_on=daily_stats['datetime'].dt.date, how='left')
dataset.drop('key_0', axis=1, inplace=True)

In [124]:
dataset['datetime_x'] = pd.to_datetime(dataset['datetime_x'])
dataset.set_index('datetime_x', inplace=True)

In [ ]:
import pvlib
#surface_tilt = 30 
surface_azimuth = 180
location = pvlib.location.Location(latitude=lat, longitude=lon)
solar_position = location.get_solarposition(dataset.index)

poa_irradiance = pvlib.irradiance.get_total_irradiance(
    surface_tilt,
    surface_azimuth,
    solar_position['apparent_zenith'],
    solar_position['azimuth'],
    dataset['DNI'],
    dataset['GHI'],
    dataset['DHI']
)

dataset['GTI'] = poa_irradiance['poa_global']

if "GTI" in selected_files:
    dataset[['GTI']].to_csv(
        f"{downloads_dir}/gti.csv"
    )

path_prepared = f'{TMYs_folder}/{site_name}/prepared_dataset_{site_name}.csv'

dataset.to_csv(path_prepared, index=True)

In [ ]:
import zipfile

zip_path = f"{TMYs_folder}/{site_name}/selected_downloads.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:

    for file in os.listdir(downloads_dir):

        full_path = os.path.join(downloads_dir, file)

        zipf.write(
            full_path,
            arcname=file
        )

print("ZIP created:", zip_path)

In [ ]:
update_job(
    'completed',
    f'Download package created: TMYs/{site_name}/selected_downloads.zip'
)

### Tmy Generation